In [0]:
%run ./lib/00_common

In [0]:
%run ./lib/02_delta_writer

In [0]:
%run ./lib/03_logger_utils

In [0]:
%run ./lib/04_quality_pandas

In [0]:
%run ./lib/06_alerting

In [0]:
import pandas as pd

run_id = get_text_widget("run_id", "").strip()
run_date = get_text_widget("run_date", "").strip()

if not run_id:
    raise ValueError("run_id es obligatorio")
if not run_date:
    raise ValueError("run_date es obligatorio")

alert_email_to = get_text_widget("alert_email_to", "")
gmail_smtp_user = get_text_widget("gmail_smtp_user", "")
gmail_app_password = get_text_widget("gmail_app_password", "")

In [0]:
try:
    stage = "gold_publish"
    dataset = "gold_tables"

    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        status="STARTED",
        message="Inicia publicación Gold con pandas-first"
    )

    silver_orders_pdf = pd.read_csv(assert_file_exists(silver_orders_current_file()))
    silver_customers_pdf = pd.read_csv(assert_file_exists(silver_customers_current_file()))

    if silver_orders_pdf.empty:
        raise Exception("silver_orders snapshot está vacío")
    if silver_customers_pdf.empty:
        raise Exception("silver_customers snapshot está vacío")

    gold_daily_pdf = build_gold_daily_sales_pdf(silver_orders_pdf)
    gold_country_pdf = build_gold_sales_by_country_pdf(silver_orders_pdf, silver_customers_pdf)

    write_csv(gold_daily_pdf, gold_daily_current_file())
    write_csv(gold_country_pdf, gold_country_current_file())

    write_gold_daily_delta(gold_daily_pdf, mode="overwrite")
    write_gold_country_delta(gold_country_pdf, mode="overwrite")

    current_rows = gold_daily_pdf.loc[
        pd.to_datetime(gold_daily_pdf["order_date"]).dt.strftime("%Y-%m-%d") == run_date
    ]

    if gold_daily_pdf.empty:
        raise Exception("gold_daily_sales quedó vacía")
    if current_rows.empty:
        raise Exception(f"No se generó fila Gold para run_date={run_date}")

    anomaly = detect_revenue_drop(gold_daily_pdf, run_date)
    if anomaly:
        create_alert_record(
            spark=spark,
            run_id=run_id,
            alert_type="REVENUE_DROP",
            severity="MEDIUM",
            message=(
                f"Revenue drop detectado. run_date={anomaly['current_run_date']}, "
                f"current_revenue={anomaly['current_revenue']}, "
                f"avg_prev_7={anomaly['avg_prev_7']}, ratio={anomaly['ratio']}"
            )
        )
        log_event(
            spark=spark,
            level="WARN",
            run_id=run_id,
            stage=stage,
            dataset=dataset,
            status="ALERT",
            message="Se detectó caída de revenue frente al promedio previo",
            extra=anomaly
        )

    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        status="OK",
        message="Gold publicada correctamente",
        rows_ok=len(gold_daily_pdf),
        extra={"run_date": run_date, "rows_for_run_date": len(current_rows)}
    )

except Exception as e:
    notify_failure(
        spark=spark,
        run_id=run_id,
        stage="gold_publish",
        dataset="gold_tables",
        error_message=str(e),
        error_code="GLD_001",
        smtp_user=gmail_smtp_user,
        smtp_app_password=gmail_app_password,
        to_email=alert_email_to
    )
    raise

print(f"Gold pandas-first OK. run_id={run_id}")